# NOM / IR / PE — Domain Expansion: Trichotomous
3 agents, 4 items, ordinal preferences, ε(3)=0

In [6]:
# ── Colab セットアップ（最初に1回だけ実行）──────────────────────────
!git clone https://github.com/shiro-seminar/NOM-matching.git

import sys
sys.path.insert(0, "/content/NOM-matching")

fatal: destination path 'NOM-matching' already exists and is not an empty directory.


In [ ]:
# コードを更新したいときはここを実行
!git -C /content/NOM-matching pull

Already up to date.


In [2]:
import torch
from domain_expansion_experiments.config      import Config
from domain_expansion_experiments.domains     import DOMAINS
from domain_expansion_experiments.train       import train
from domain_expansion_experiments.evaluate    import evaluate_mechanism, print_table
from domain_expansion_experiments.benchmarks  import BENCHMARKS
from domain_expansion_experiments.data_gen    import sample_batch
from domain_expansion_experiments.model       import AllocationNet
from domain_expansion_experiments.allocations import ir_pe_mask

## 1. Config

In [3]:
DOMAIN = "trichotomous_extended_e3"   # trichotomous / trichotomous_extended_e3 / four_chotomous_e4 / strict

cfg = Config(
    domain     = DOMAIN,
    steps      = 50_000,
    S          = 8,
    M          = 8,
    batch_size = 64,
    device     = "cuda" if torch.cuda.is_available() else "cpu",
    seed       = 42,
)
print(cfg)
print(f"device: {cfg.device}  num_ranks: {cfg.num_ranks}")

Config(num_agents=3, num_items=4, domain='trichotomous_extended_e3', num_ranks=4, hidden=256, depth=4, batch_size=64, steps=50000, lr=0.0003, grad_clip=1.0, seed=42, S=8, M=8, temperature=1.0, lambda_nom=0.0, rho=1.0, rho_mult=1.005, rho_max=200.0, dual_update_every=100, nom_target=0.005, welfare_weight=0.02, device='cuda')
device: cuda  num_ranks: 4


In [ ]:
import os
import time
import torch
import torch.nn as nn
# 元の train.py で使われている損失関数を追加でインポート
from domain_expansion_experiments.losses import augmented_objective

def train_with_checkpoint(cfg: Config | None = None, verbose: bool = True, checkpoint_path: str = None) -> AllocationNet:
    if cfg is None:
        cfg = Config()
    torch.manual_seed(cfg.seed)
    domain = DOMAINS[cfg.domain]
    device = torch.device(cfg.device)

    net = AllocationNet(cfg).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=cfg.lr)
    t0  = time.time()

    start_step = 1

    # ─── チェックポイントの読み込み（レジューム処理） ───
    if checkpoint_path and os.path.exists(checkpoint_path):
        print(f"重大な切断からの再開: {checkpoint_path} を読み込みます...")
        checkpoint = torch.load(checkpoint_path, map_location=device)

        net.load_state_dict(checkpoint["state_dict"])
        opt.load_state_dict(checkpoint["optimizer_state_dict"])
        start_step = checkpoint["step"] + 1

        # 動的に変わる設定値も復元
        cfg.lambda_nom = checkpoint["lambda_nom"]
        cfg.rho = checkpoint["rho"]
        print(f"-> step={start_step} から学習を再開します。 (lambda={cfg.lambda_nom:.4f}, rho={cfg.rho:.2f})")
    else:
        print("チェックポイントが見つからないか、指定されていません。最初から学習を開始します。")

    # ─── 学習ループ ───
    for step in range(start_step, cfg.steps + 1):
        batch         = sample_batch(cfg)
        marginal_rank = batch["marginal_rank"]
        endow_idx     = batch["endow_idx"]
        S             = batch["S"]
        mask          = ir_pe_mask(cfg, S, endow_idx)

        loss, stats = augmented_objective(cfg, domain, net,
                                          marginal_rank, endow_idx, S, mask)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), cfg.grad_clip)
        opt.step()

        if step % cfg.dual_update_every == 0:
            nom_val = stats["nom"]
            cfg.lambda_nom = max(0.0, cfg.lambda_nom + cfg.rho * nom_val)
            if nom_val > cfg.nom_target:
                cfg.rho = min(cfg.rho * cfg.rho_mult, cfg.rho_max)

        if verbose and (step % 200 == 0 or step == 1):
            print(
                f"  step={step:6d}  welfare={stats['welfare']:.4f}  "
                f"nom={stats['nom']:.5f}  lambda={cfg.lambda_nom:.3f}  "
                f"rho={cfg.rho:.2f}  elapsed={time.time()-t0:.0f}s"
            )

        # ─── 定期保存（例: 2000ステップごと、および最終ステップ） ───
        if checkpoint_path and (step % 2000 == 0 or step == cfg.steps):
            # 一時的なエラーで保存が失敗してファイルが壊れるのを防ぐため、別名で保存してリネーム
            tmp_path = checkpoint_path + ".tmp"
            torch.save({
                "step": step,
                "state_dict": net.state_dict(),
                "optimizer_state_dict": opt.state_dict(),
                "lambda_nom": cfg.lambda_nom,
                "rho": cfg.rho,
                "cfg_dict": cfg.__dict__
            }, tmp_path)
            os.replace(tmp_path, checkpoint_path)
            print(f"  >> [Checkpoint Saved] step={step} を Google Drive に保存しました。")

    return net

## 2. Train

In [4]:
# Google Drive上の保存先パス（切断されても消えない場所）
CKPT_PATH = f"/content/drive/MyDrive/{DOMAIN}_checkpoint.pt"

# 自前で作ったチェックポイント対応の学習関数を実行
net = train_with_checkpoint(cfg, verbose=True, checkpoint_path=CKPT_PATH)
net.eval()

  step=     1  welfare=-2.9959  nom=0.01273  lambda=0.000  rho=1.00  elapsed=1s
  step=   200  welfare=-3.0469  nom=0.02357  lambda=0.045  rho=1.01  elapsed=13s
  step=   400  welfare=-3.1247  nom=0.01302  lambda=0.068  rho=1.02  elapsed=25s
  step=   600  welfare=-2.9871  nom=0.00512  lambda=0.081  rho=1.03  elapsed=37s
  step=   800  welfare=-2.8755  nom=0.00847  lambda=0.094  rho=1.04  elapsed=48s
  step=  1000  welfare=-2.5661  nom=0.03032  lambda=0.126  rho=1.04  elapsed=59s
  step=  1200  welfare=-3.0000  nom=0.00895  lambda=0.148  rho=1.05  elapsed=71s
  step=  1400  welfare=-3.1513  nom=0.01090  lambda=0.167  rho=1.06  elapsed=82s
  step=  1600  welfare=-2.9221  nom=0.02029  lambda=0.210  rho=1.07  elapsed=94s
  step=  1800  welfare=-2.8750  nom=0.01570  lambda=0.229  rho=1.08  elapsed=105s
  step=  2000  welfare=-3.2571  nom=0.01042  lambda=0.266  rho=1.09  elapsed=116s
  step=  2200  welfare=-2.7362  nom=0.00842  lambda=0.280  rho=1.09  elapsed=128s
  step=  2400  welfare=-3.

KeyboardInterrupt: 

## 3. Evaluate — benchmarks vs learned net

In [ ]:
EVAL_N  = 500    # evaluation sample size
EVAL_S  = 64     # NOM opponent samples
EVAL_M  = 64     # NOM misreport samples

eval_cfg = Config(domain=DOMAIN, batch_size=EVAL_N, device=cfg.device, seed=0)
torch.manual_seed(0)

domain = DOMAINS[DOMAIN]
batch  = sample_batch(eval_cfg)
marginal_rank = batch["marginal_rank"]
endow_idx     = batch["endow_idx"]
S             = batch["S"]

# WMAX: oracle upper bound on welfare (IR+PE constraint)
irpe_m = ir_pe_mask(eval_cfg, S, endow_idx)
wmax_s = (S.sum(1) + (1 - irpe_m) * (-1e9)).max(1).values

results = []
for bname, bfn in BENCHMARKS.items():
    print(f"Evaluating {bname}...")
    r = evaluate_mechanism(bname, bfn, eval_cfg, domain,
                           marginal_rank, endow_idx, S, wmax_s,
                           eval_S=EVAL_S, eval_M=EVAL_M)
    results.append(r)

def net_mech(cfg_, mr_, ei_, S_):
    mask_ = ir_pe_mask(cfg_, S_, ei_)
    return net(mr_, mask=mask_, temperature=1e-3)

results.append(evaluate_mechanism("LearnedNet", net_mech, eval_cfg, domain,
                                  marginal_rank, endow_idx, S, wmax_s,
                                  eval_S=EVAL_S, eval_M=EVAL_M))

print_table(results)

Evaluating Endowment...
Evaluating WMAX-IR...
Evaluating WMAX-PE...
Evaluating WMAX-IR-PE...
Evaluating TTC-ordinal...
Evaluating Random-IR-PE...

-------------------------------------------------------------------------------------
Mechanism             Welfare  W/WMAX  IR-viol%     PE%   IR+PE%  NOM-mean  NOM-viol%
-------------------------------------------------------------------------------------
Endowment             -1.9780   1.427       0.0    26.0     57.6   0.00000        0.0
WMAX-IR               -0.8860   0.639       0.0   100.0     17.8   0.00000        0.0
WMAX-PE               -0.8860   0.639      22.0   100.0     11.0   0.00000        0.0
WMAX-IR-PE            -1.3860   1.000       0.0    68.4    100.0   0.00000        0.0
TTC-ordinal           -1.0140   0.732       0.0    88.4     25.0   0.00000        0.0
Random-IR-PE          -1.3860   1.000       0.0    68.4    100.0   0.00000        0.0
LearnedNet            -1.3860   1.000       0.0    68.4    100.0   0.00000     

## 4. (Optional) チェックポイントを保存 / 読み込み

In [ ]:
# Google Drive にモデルを保存したい場合
from google.colab import drive
drive.mount("/content/drive")

import shutil
shutil.copy(f"{DOMAIN}_net.pt", f"/content/drive/MyDrive/{DOMAIN}_net.pt")
print(f"saved: {DOMAIN}_net.pt")

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: 'trichotomous_net.pt'

In [ ]:
# Drive から読み込む場合
CKPT = f"/content/drive/MyDrive/{DOMAIN}_net.pt"
ckpt = torch.load(CKPT, map_location=cfg.device)
net2 = AllocationNet(Config(**ckpt["cfg"]))
net2.load_state_dict(ckpt["state_dict"])
net2.eval()
print(f"loaded: step={ckpt.get('step', 'final')}")

In [5]:
!cat /content/NOM-matching/domain_expansion_experiments/train.py

from __future__ import annotations
import time
import torch
import torch.nn as nn
from .config import Config
from .domains import DOMAINS
from .allocations import ir_pe_mask
from .data_gen import sample_batch
from .model import AllocationNet
from .losses import augmented_objective


def train(cfg: Config | None = None, verbose: bool = True) -> AllocationNet:
    if cfg is None:
        cfg = Config()
    torch.manual_seed(cfg.seed)
    domain = DOMAINS[cfg.domain]
    device = torch.device(cfg.device)

    net = AllocationNet(cfg).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=cfg.lr)
    t0  = time.time()

    for step in range(1, cfg.steps + 1):
        batch         = sample_batch(cfg)
        marginal_rank = batch["marginal_rank"]
        endow_idx     = batch["endow_idx"]
        S             = batch["S"]
        mask          = ir_pe_mask(cfg, S, endow_idx)

        loss, stats = augmented_objective(cfg, domain, net,
                                          marginal